# Basic Statistics-2

## Problem Statement : Hospital Patient Data Analysis

In [1]:
import pandas as pd
import numpy as np

##### 1.Load the patient dataset and show summary with info().

In [13]:
data = pd.read_csv('Patient_Data.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


##### 2.Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [14]:
billing_col = data[['PatientID', 'Department', 'Doctor', 'BillAmount']]
print('/n Billing related columns: ')
print(billing_col.head())

/n Billing related columns: 
   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN


##### 3.Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [15]:
data = data.drop(columns = ['ReceptionistID', 'CheckInTime'])
data

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


##### 4.Use groupby to find total bill amount per department.

In [16]:
tot_bill = data.groupby('Department')['BillAmount'].sum()
print('/n Total bill amount per department:')
print(tot_bill)

/n Total bill amount per department:
Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


##### 5.Remove duplicate patient records based on PatientID.

In [17]:
data = data.drop_duplicates(subset = 'PatientID')
data

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [20]:
data.duplicated().sum()

np.int64(0)

##### 6.Fill missing BillAmount values with the mean bill amount.

In [22]:
mean_bill = data['BillAmount'].mean()
data['BillAmount'] = data['BillAmount'].fillna(mean_bill)
data

C:\Users\h238p\AppData\Local\Temp\ipykernel_13232\3020087054.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['BillAmount'] = data['BillAmount'].fillna(mean_bill)


,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


In [23]:
data.isnull().sum()

PatientID     0
Name          0
Department    0
Doctor        0
BillAmount    0
dtype: int64

##### 7.Merge the billing dataset with patient dataset on PatientID.

In [24]:
billing_data = pd.read_csv('Billing_Data.csv')
merge_data = pd.merge(data, billing_data, on = 'PatientID', how = 'inner')
print('/n Merged Data : ')
print(merge_data.head())

/n Merged Data : 
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


##### 8.Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [25]:
new_patients = pd.DataFrame({
    'PatientID': [201, 202],
    'Department':['Cardiology', 'Neurology'],
    'Doctor':['Dr.Kumar', 'Dr.Rao'],
    'BillAmount':[4500, 6000]
})

combined_data = pd.concat([merge_data, new_patients],axis = 0, ignore_index =True)

print('/nAfter adding new patients:')
print(combined_data)

/nAfter adding new patients:
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000            2000.0   
1        102      Bob    Neurology   Dr. John  6233.333333            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith  6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose  6233.333333            1000.0   
5        201      NaN   Cardiology   Dr.Kumar  4500.000000               NaN   
6        202      NaN    Neurology     Dr.Rao  6000.000000               NaN   

   FinalAmount  
0       3000.0  
1       3500.0  
2       5000.0  
3       3200.0  
4       4000.0  
5          NaN  
6          NaN  


##### 9.Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [26]:
billing_cat_data = pd.DataFrame({
    'InsuranceCovered':[1000]*len(combined_data),
    'FinalAmount':[4000]*len(combined_data)
})

final_data = pd.concat([combined_data, billing_cat_data], axis = 1)

print('/nFinal data:')
print(final_data)

/nFinal data:
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000            2000.0   
1        102      Bob    Neurology   Dr. John  6233.333333            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith  6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose  6233.333333            1000.0   
5        201      NaN   Cardiology   Dr.Kumar  4500.000000               NaN   
6        202      NaN    Neurology     Dr.Rao  6000.000000               NaN   

   FinalAmount  InsuranceCovered  FinalAmount  
0       3000.0              1000         4000  
1       3500.0              1000         4000  
2       5000.0              1000         4000  
3       3200.0              1000         4000  
4       4000.0              1000         4000  
5          NaN              1000         4000  
6        